# 09 - Feature Importance Stability Analysis

Assess the stability of SHAP-based feature importance rankings across
cross-validation folds.

**Method**: Within each of 5 CV folds, train GradientBoosting on combined
features, compute SHAP values on the test set, and derive mean |SHAP|
per feature. Then compute pairwise Spearman rank correlations between
all 10 fold pairs.

**Expected results**:
- Mean Spearman rho = 0.820
- Ki67 and age consistently rank in top 2 across all 5 folds

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
from src.models import get_classifiers

## 1. Prepare Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

gse_exp_norm = zscore_normalize(gse_exp)
pathway_features = compute_pathway_scores(gse_exp_norm)
pathway_features = add_ratio_features(pathway_features)
clinical_features = encode_clinical_features(gse_clin)
X = build_feature_matrix(pathway_features, clinical_features)
y = gse_clin['high_risk'].values

feature_names = X.columns.tolist()
print(f"Features: {len(feature_names)}")

## 2. Per-Fold SHAP Importance

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_arr = np.array(X)
gb_template = get_classifiers()['Gradient Boosting']

fold_importances = []  # List of mean |SHAP| per feature per fold

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_arr, y)):
    print(f"Fold {fold_idx + 1}/5...")
    
    X_train, X_test = X_arr[train_idx], X_arr[test_idx]
    y_train = y[train_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    from sklearn.base import clone
    gb = clone(gb_template)
    gb.fit(X_train_s, y_train)
    
    explainer = shap.TreeExplainer(gb)
    shap_values = explainer.shap_values(X_test_s)
    
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    fold_importances.append(mean_abs_shap)

importance_df = pd.DataFrame(fold_importances, columns=feature_names,
                              index=[f'Fold {i+1}' for i in range(5)])
print("\nMean |SHAP| per fold:")
print(importance_df.round(4).to_string())

## 3. Spearman Rank Correlation

In [ ]:
# Compute ranks per fold
rank_df = importance_df.rank(axis=1, ascending=False)

# Pairwise Spearman correlations
rhos = []
for i in range(5):
    for j in range(i + 1, 5):
        rho, _ = spearmanr(fold_importances[i], fold_importances[j])
        rhos.append(rho)

mean_rho = np.mean(rhos)
print(f"Pairwise Spearman correlations: {[f'{r:.3f}' for r in rhos]}")
print(f"\nMean Spearman rho: {mean_rho:.3f}")

# Check top-ranked features across folds
print("\nTop 3 features per fold:")
for fold in rank_df.index:
    top3 = rank_df.loc[fold].sort_values().head(3).index.tolist()
    print(f"  {fold}: {', '.join(top3)}")

## 4. Stability Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (A) Heatmap of SHAP importance across folds
ax1 = axes[0]
# Normalize for visualization
norm_importance = importance_df.div(importance_df.sum(axis=1), axis=0)
sns.heatmap(norm_importance.T, annot=True, fmt='.3f', cmap='YlOrRd',
            ax=ax1, cbar_kws={'label': 'Normalized Mean |SHAP|'})
ax1.set_title('(A) SHAP Feature Importance Across Folds', fontsize=13)
ax1.set_ylabel('Feature')
ax1.set_xlabel('CV Fold')

# (B) Boxplot of feature ranks
ax2 = axes[1]
rank_melted = rank_df.T
rank_melted['Feature'] = rank_melted.index
rank_melted = rank_melted.melt(id_vars='Feature', var_name='Fold', value_name='Rank')

# Sort by median rank
median_ranks = rank_df.median().sort_values()
order = median_ranks.index.tolist()

sns.boxplot(data=rank_melted, y='Feature', x='Rank', order=order, ax=ax2, orient='h')
ax2.set_title(f'(B) Feature Rank Stability (Mean Spearman \u03C1 = {mean_rho:.3f})', fontsize=13)
ax2.set_xlabel('Importance Rank (1 = most important)')

plt.tight_layout()
plt.savefig('../figures/fig_stability.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_stability.png")
plt.show()